# Analyze fitness effects by subset

Compare amino-acid fitness effects between subsets (host, geographic, temporal) using scatter plots.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from scipy.stats import pearsonr

sns.set_theme(font_scale=1.0, style='ticks', palette='colorblind')

In [ ]:
# Parameters
counts_cutoff = 10  # minimum actual or expected counts in BOTH subsets

In [ ]:
# Read subset fitness effects
df = pd.read_csv('../results/subset_aa_fitness_effects.csv')

# Filter to nonsynonymous mutations
df = df[df['mut_class'] == 'nonsynonymous'].copy()
print(f'Nonsynonymous AA mutations: {len(df):,}')
df.head()

In [ ]:
# Define mutation identity columns for matching across subsets
id_cols = ['subtype', 'segment', 'gene', 'codon_site', 'wt_aa', 'mut_aa']

# Define subset pairs to compare, organized by type
subset_pairs_by_type = {}
for subset_type, type_df in df.groupby('subset_type'):
    subsets = sorted(type_df['subset'].unique())
    pairs = list(combinations(subsets, 2))
    if pairs:
        subset_pairs_by_type[subset_type] = pairs

print('Subset pairs to compare:')
for subset_type, pairs in subset_pairs_by_type.items():
    for a, b in pairs:
        print(f'  {subset_type}: {a} vs {b}')

In [ ]:
def make_scatter(df, subset_a, subset_b, id_cols, counts_cutoff, ax):
    """Make a scatter plot comparing fitness effects between two subsets."""
    df_a = df[df['subset'] == subset_a].set_index(id_cols)
    df_b = df[df['subset'] == subset_b].set_index(id_cols)

    # Inner join on mutation identity
    merged = df_a[['delta_fitness', 'actual_count', 'expected_count']].join(
        df_b[['delta_fitness', 'actual_count', 'expected_count']],
        lsuffix=f'_{subset_a}', rsuffix=f'_{subset_b}',
        how='inner'
    ).dropna()

    # Filter: require at least counts_cutoff in both subsets
    merged = merged[
        ((merged[f'actual_count_{subset_a}'] >= counts_cutoff) |
         (merged[f'expected_count_{subset_a}'] >= counts_cutoff)) &
        ((merged[f'actual_count_{subset_b}'] >= counts_cutoff) |
         (merged[f'expected_count_{subset_b}'] >= counts_cutoff))
    ]

    if len(merged) < 2:
        ax.set_title(f'{subset_a} vs {subset_b}\nN = {len(merged)} (too few)')
        return

    x = merged[f'delta_fitness_{subset_a}']
    y = merged[f'delta_fitness_{subset_b}']
    r, p = pearsonr(x, y)

    ax.scatter(x, y, alpha=0.3, s=8, rasterized=True)

    # Add diagonal reference line
    lims = [min(x.min(), y.min()), max(x.max(), y.max())]
    ax.plot(lims, lims, 'k--', alpha=0.3, lw=0.8)

    ax.set_xlabel(f'{subset_a}')
    ax.set_ylabel(f'{subset_b}')
    ax.set_title(f'R = {r:.3f}, N = {len(merged):,}')
    ax.set_aspect('equal')

In [ ]:
# Make scatter plots for each subset type
for subset_type, pairs in subset_pairs_by_type.items():
    n_pairs = len(pairs)
    fig, axs = plt.subplots(1, n_pairs, figsize=(5 * n_pairs, 5), squeeze=False)
    axs = axs[0]

    for i, (subset_a, subset_b) in enumerate(pairs):
        make_scatter(df, subset_a, subset_b, id_cols, counts_cutoff, axs[i])

    fig.suptitle(
        f'{subset_type.capitalize()} subsets: fitness effect correlation\n'
        f'(nonsynonymous, counts >= {counts_cutoff})',
        fontsize=13
    )
    plt.tight_layout()
    sns.despine()
    plt.show()